In [22]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("pypdf").setLevel(logging.ERROR)

### Part 1 - Data Ingestion

In [1]:
from langchain_core.documents import Document

In [4]:
doc = Document(
    page_content = "this is the main text content I am using to create my RAG pipeline",
    metadata = {
        "source":"example.txt",
        "author": "mumuksha pant",
        "date_created":"2026-06-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'author': 'mumuksha pant', 'date_created': '2026-06-01'}, page_content='this is the main text content I am using to create my RAG pipeline')

In [5]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [6]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",

    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems


    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [23]:
### TextLoader
'''
TextLoader in LangChain is used to read text files and convert them into LangChain Document objects.
It loads the file's content and stores it in a format that LangChain can process for tasks like chunking, embedding, retrieval, and question answering.
'''
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")
document=loader.load()
print(document)

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [24]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True

)

documents=dir_loader.load()
documents

100%|██████████| 2/2 [00:00<00:00, 1014.83it/s]


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/text_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervise

In [ ]:
# Working with PDFs

In [25]:
# PyPDFLoader creates one Document per page- not per file 
import textwrap
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader

# Load all the PDF files from the directory
dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",        # Pattern to match files
    loader_cls=PyPDFLoader,  # Loader class to use
    show_progress=False,
)
pdf_documents = dir_loader.load()

# Print them readably
print(f"Loaded {len(pdf_documents)} document(s)\n")

for index, document in enumerate(pdf_documents):
    source = document.metadata.get("source", "unknown")
    page = document.metadata.get("page", "?")
    print("=" * 80)
    print(f"[{index}] Source: {source}  |  Page: {page}  |  {len(document.page_content)} chars")
    print("-" * 80)
    print(textwrap.fill(document.page_content[:500], width=80))
    print()

Loaded 28 document(s)

[0] Source: ../data/pdf/paper.pdf  |  Page: 0  |  2860 chars
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain
avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗
Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com
Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of
Toronto aidan@cs.toronto.edu Łukasz

[1] Source: ../data/pdf/paper.pdf  |  Page: 1  |  4258 chars
--------------------------------------------------------------------------------
1 Introduction Recurrent neural networks, long short-term memory [13] and gated
recurrent [7] neural networks in particular, have been firmly established as
state of the art approaches in sequ

In [27]:
pdf_documents[0]

### PART 2 - Chunking